Generate frames for annotation

Imports

In [15]:
import ffmpeg
import os
import random
import subprocess
import numpy as np
import cv2

Looping through every video stored in VODS, scrpit extracts 100 frames with a normal distribrution to be more efficient with getting the "meat" from the middle instead of caster segments + intro. Frames stores in "Frames" and list of videos already sampled stored in "sampled_videos.txt".

In [16]:
# Define input folder containing videos
video_folder = "VODS"
log_file = "sampled_videos.txt"
output_folder = "Frames"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Load processed videos from log file
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        processed_videos = set(f.read().splitlines())
else:
    processed_videos = set()

# Get list of videos in the folder
video_files = [f for f in os.listdir(video_folder) if f.endswith((".mp4", ".webm"))]

for video in video_files:
    video_path = os.path.join(video_folder, video)

    # Skip if already processed
    if video in processed_videos:
        print(f"Skipping {video}, already processed.")
        continue

    # Step 1: Get video duration
    cmd = f'ffprobe -i "{video_path}" -show_entries format=duration -v quiet -of csv="p=0"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    try:
        video_length = float(result.stdout.strip())
    except ValueError:
        print(f"Could not determine duration for {video}, skipping.")
        continue

    # Step 2: Generate 100 timestamps using normal distribution
    mu = video_length / 2  # Mean at the middle
    sigma = video_length / 4  # Adjust std to keep values within range

    timestamps = np.random.normal(mu, sigma, 100)
    timestamps = np.clip(timestamps, 0, video_length)  # Keep within video bounds
    timestamps = np.sort(timestamps)  # Sort to maintain chronological order

    # Step 3: Extract frames at these timestamps using ffmpeg
    video_name = os.path.splitext(video)[0]  # Remove extension
    video_output_folder = os.path.join(output_folder, video_name)
    os.makedirs(video_output_folder, exist_ok=True)

    for i, timestamp in enumerate(timestamps):
        output_image = os.path.join(video_output_folder, f"frame_{i:03d}.png")
        cmd = f'ffmpeg -ss {timestamp:.2f} -i "{video_path}" -frames:v 1 "{output_image}" -y -hide_banner -loglevel error'
        subprocess.run(cmd, shell=True)

    # Step 4: Log the processed video
    with open(log_file, "a") as f:
        f.write(video + "\n")

    print(f"Processed {video} successfully!")

print("All videos processed.")


Skipping G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04.webm, already processed.
Skipping G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 05.mp4, already processed.
All videos processed.


Now we will crop the images to only include mini-map (1), time + score + bomb/start (2), killfeed (3).

In [17]:
# We perform the cropping step after so we can do image classification on the whole frame, then crop when we want the info.

# Define the parent folder containing video subfolders
frames_folder = "Frames"

# Define crop areas (x, y, width, height) with descriptive names
crop_areas = {
    "minimap": (20, 30, 300, 275),
    "score": (360, 0, 500, 150),
    "killfeed": (850, 50, 430, 350)
}

# Loop through each video folder inside "frames"
for video_folder in os.listdir(frames_folder):
    video_folder_path = os.path.join(frames_folder, video_folder)
    
    # Ensure it's a directory
    if not os.path.isdir(video_folder_path):
        continue

    print(f"Processing video: {video_folder}")

    # Loop through all images in the folder
    for img_file in os.listdir(video_folder_path):
        if not img_file.endswith((".png", ".jpg", ".jpeg")):
            continue  # Skip non-image files

        # Extract frame number from filename
        frame_number = os.path.splitext(img_file)[0]
        img_path = os.path.join(video_folder_path, img_file)

        # Check if all cropped images exist before loading the image
        missing_crops = []
        for crop_name in crop_areas.keys():
            cropped_filename = f"{frame_number}_{crop_name}.png"
            cropped_path = os.path.join(video_folder_path, cropped_filename)
            if not os.path.exists(cropped_path):
                missing_crops.append((crop_name, cropped_path))  # Store missing ones

        # Skip processing if all crops exist
        if not missing_crops:
            print(f"Skipping {img_file}, all cropped frames exist.")
            continue

        # Load the image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Skipping {img_file}, could not load.")
            continue

        # Perform classification (Placeholder)
        # TODO: Replace with actual classification function
        print(f"Classifying {img_file}...")  
        # classification_result = classify_image(img)  # Example function

        # Apply cropping only for missing crops
        for crop_name, cropped_path in missing_crops:
            x, y, w, h = crop_areas[crop_name]

            if x + w > img.shape[1] or y + h > img.shape[0]:
                print(f"Skipping {crop_name} crop for {img_file}, out of bounds.")
                continue

            cropped_img = img[y:y+h, x:x+w]
            cv2.imwrite(cropped_path, cropped_img)
            print(f"Saved cropped image: {cropped_path}")

print("Processing complete!")



Processing video: G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04
Skipping frame_000.png, all cropped frames exist.
Classifying frame_000_killfeed.png...
Saved cropped image: Frames\G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04\frame_000_killfeed_minimap.png
Skipping score crop for frame_000_killfeed.png, out of bounds.
Skipping killfeed crop for frame_000_killfeed.png, out of bounds.
Classifying frame_000_minimap.png...
Skipping minimap crop for frame_000_minimap.png, out of bounds.
Skipping score crop for frame_000_minimap.png, out of bounds.
Skipping killfeed crop for frame_000_minimap.png, out of bounds.
Classifying frame_000_score.png...
Skipping minimap crop for frame_000_score.png, out of bounds.
Skipping score crop for frame_000_score.png, out of bounds.
Skipping killfeed crop for frame_000_score.png, out of bounds.
Skipping frame_001.png, all cropped frames exist.
Classifying frame_001_killfeed.png...
Saved cropped image: Frames\G2 vs. SEN  - VCT Americas Kickoff -